In [1]:
    #Step1
    import pandas as pd
    import pandasql as ps
    import pixiedust
    import sys
    sys.path.append('..')
    import util
    import etl
    import pyarrow.parquet as pq
    import pyarrow as pa
    !pip install duckdb
    import duckdb
    import inspect

    # Get the path of the imported module (etl.py)
    etl_module_path = inspect.getfile(etl)
    print("Path of the imported ETL module (etl.py):", etl_module_path)
    util.usedatabase(spark, "real_world_data_jun_2022")

    pixiedust.enableJobMonitor()

    con = duckdb.connect()

Pixiedust database opened successfully
Table VERSION_TRACKER created successfully
Table METRICS_TRACKER created successfully

Share anonymous install statistics? (opt-out instructions)

PixieDust will record metadata on its environment the next time the package is installed or updated. The data is anonymized and aggregated to help plan for future releases, and records only the following values:

{
   "data_sent": currentDate,
   "runtime": "python",
   "application_version": currentPixiedustVersion,
   "space_id": nonIdentifyingUniqueId,
   "config": {
       "repository_id": "https://github.com/ibm-watson-data-lab/pixiedust",
       "target_runtimes": ["Data Science Experience"],
       "event_id": "web",
       "event_organizer": "dev-journeys"
   }
}
You can opt out by calling pixiedust.optOut() in a new cell.


Pixiedust runtime updated. Please restart kernel
Table SPARK_PACKAGES created successfully
Table USER_PREFERENCES created successfully
Table service_connections created successfully
Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
     |████████████████████████████████| 20.2 MB 7.3 MB/s eta 0:00:01
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [3]:
Epilepsy_Combined = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-superset-finalV1")

In [8]:
Epilepsy_Combined = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Smallset_scaleddf")

In [3]:
Epilepsy_Combined = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-smallset-finalF1_Numbered_NoNullStr.parquet")

In [4]:
balanced_df = Epilepsy_Combined.drop("personid")

In [5]:
# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions = balanced_df.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    balanced_df = balanced_df.repartition(num_partitions)
    print(f"DataFrame repartitioned into {num_partitions} partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")

DataFrame repartitioned into 14 partitions successfully.


In [21]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
import random
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, VectorSizeHint

# Step 0: Mimicking randomSplit with seed and randomness
def probabilistic_split(df: DataFrame, fractions: list, seed=None) -> list:
    if seed is not None:
        random.seed(seed)

    cumulative_fractions = [sum(fractions[:i + 1]) for i in range(len(fractions))]
    random_col = F.rand(seed)
    df_with_random = df.withColumn("random", random_col)
    splits = []
    prev_fraction = 0
    for fraction in cumulative_fractions:
        split_df = df_with_random.filter((F.col("random") >= prev_fraction) & (F.col("random") < fraction))
        splits.append(split_df.drop("random"))
        prev_fraction = fraction
    return splits

# Train, validation, and test sampling fractions
train_fraction = 0.7
valid_fraction = 0.2
test_fraction = 0.1
fractions = [train_fraction, valid_fraction, test_fraction]
seed_value = 23
train_data, valid_data, test_data = probabilistic_split(balanced_df, fractions, seed=seed_value)

# Show class distribution in train, validation, and test datasets
train_data.groupBy('label').count().show()
valid_data.groupBy('label').count().show()
test_data.groupBy('label').count().show()

# Print counts
print("Sampled data count:", balanced_df.count())
print("Train data count:", train_data.count())
print("Validation data count:", valid_data.count())
print("Test data count:", test_data.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Test data count: 116649


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
columns = train_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

▸,:,


In [ ]:
columns = valid_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

In [ ]:
columns = test_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

In [23]:
# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions = train_data.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    train_data = train_data.repartition(num_partitions)
    print(f"DataFrame repartitioned into {num_partitions} partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")
    
# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions1 = valid_data.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    valid_data = valid_data.repartition(num_partitions1)
    print(f"DataFrame repartitioned into {num_partitions1} partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")
    
# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions2 = test_data.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    test_data = test_data.repartition(num_partitions2)
    print(f"DataFrame repartitioned into {num_partitions2} partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")

▸,:,


DataFrame repartitioned into 14 partitions successfully.
DataFrame repartitioned into 14 partitions successfully.
DataFrame repartitioned into 14 partitions successfully.


In [12]:
# from pyspark.sql import DataFrame
# from pyspark.ml import Pipeline
# from pyspark.ml.feature import VectorAssembler, VectorSizeHint
# # Initialize stages list for preprocessing pipeline
# stages_preprocess = []
# # Step 1: Define vector columns and non-vector columns based on schema
# vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
# non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# # Step 2: Function to apply VectorSizeHint and transform data
# def apply_vector_size_hint(data, col_name):
#     # Sample a small portion of the data to determine vector size
#     sample_fraction = 0.0001  # Using 1% of the data for sampling
#     sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
#     sample_row = sampled_df.take(1)

#     if sample_row:
#         vector_size = len(sample_row[0][col_name])
#         print(f"Column '{col_name}' vector size: {vector_size}")

#         # Apply VectorSizeHint if the vector size is greater than 0
#         if vector_size > 0:
#             size_hint = VectorSizeHint(inputCol=col_name, size=vector_size)
#             pipeline_size_hint = Pipeline(stages=[size_hint])
            
#             print(f"Fitting VectorSizeHint for column '{col_name}' on sample of {sample_fraction * 100}% of data...")
#             model_size_hint = pipeline_size_hint.fit(sampled_df)
            
#             print(f"Applying VectorSizeHint for column '{col_name}' on full dataset...")
#             return model_size_hint.transform(data)
#     else:
#         print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")

#     return data

# # Applying the function to each vector column
# for col_name in vector_cols:
#     print(f"Applying VectorSizeHint to vector column: '{col_name}'")
#     train_data = apply_vector_size_hint(train_data, col_name)

# # Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
# final_input_cols = vector_cols + non_vector_cols
# print("Final input columns for feature assembly:", final_input_cols)

# # Step 5: Assemble final features column using VectorAssembler
# final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")
# stages_preprocess.append(final_assembler)
# # Apply StandardScaler
# scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
# stages_preprocess.append(scaler)
# # Create preprocessing pipeline
# preprocess_pipeline = Pipeline(stages=stages_preprocess)
# # Step 6: Fit preprocessing pipeline on train_data and transform train, valid, and test sets
# preprocess_model = preprocess_pipeline.fit(train_data)
# train_data_transformed = preprocess_model.transform(train_data)
# valid_data_transformed = preprocess_model.transform(valid_data)
# test_data_transformed = preprocess_model.transform(test_data)
# print("Preprocessing pipeline successfully applied to train, valid, and test sets.")

Applying VectorSizeHint to vector column: 'race_vector'
Column 'race_vector' vector size: 1
Fitting VectorSizeHint for column 'race_vector' on sample of 0.01% of data...
Applying VectorSizeHint for column 'race_vector' on full dataset...
Applying VectorSizeHint to vector column: 'gender_vector'
Column 'gender_vector' vector size: 1
Fitting VectorSizeHint for column 'gender_vector' on sample of 0.01% of data...
Applying VectorSizeHint for column 'gender_vector' on full dataset...
Final input columns for feature assembly: ['race_vector', 'gender_vector', 'age_of_TBI_diagnosis', 'MedicalHistory', 'E72', 'Q02', 'S37', 'R52', 'L74', 'T50', 'S87', 'S41', 'C25', 'V58', 'B37', 'T58', 'V41', 'X34', 'C86', 'I01', 'Q11', 'M75', 'B70', 'Q52', 'W27', 'R36', 'V65', 'V28', 'I47', 'K87', 'I52', 'E86', 'Z33', 'Q23', 'Q86', 'N46', 'L62', 'V97', 'T30', 'L50', 'S56', 'H62', 'C91', 'G70', 'Q45', 'T27', 'Z77', 'S51', 'P05', 'F29', 'A83', 'F20', 'W89', 'B68', 'O72', 'X77', 'L87', 'X39', 'B42', 'R59', 'V38', 

Py4JJavaError: An error occurred while calling o164.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 7 in stage 66.0 failed 1 times, most recent failure: Lost task 7.0 in stage 66.0 (TID 881, localhost, executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.nio.HeapByteBuffer.<init>(HeapByteBuffer.java:57)
	at java.nio.ByteBuffer.allocate(ByteBuffer.java:335)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator$$anonfun$5.apply(ShuffleBlockFetcherIterator.scala:458)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator$$anonfun$5.apply(ShuffleBlockFetcherIterator.scala:458)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.allocateNewChunkIfNeeded(ChunkedByteBufferOutputStream.scala:87)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.write(ChunkedByteBufferOutputStream.scala:75)
	at org.apache.spark.util.Utils$$anonfun$copyStream$1.apply$mcJ$sp(Utils.scala:363)
	at org.apache.spark.util.Utils$$anonfun$copyStream$1

In [24]:
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and transform data
def apply_vector_size_hint(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 1% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")

        # Apply VectorSizeHint if the vector size is greater than 0
        if vector_size > 0:
            size_hint = VectorSizeHint(inputCol=col_name, size=vector_size)
            pipeline_size_hint = Pipeline(stages=[size_hint])
            
            print(f"Fitting VectorSizeHint for column '{col_name}' on sample of {sample_fraction * 100}% of data...")
            model_size_hint = pipeline_size_hint.fit(sampled_df)
            
            print(f"Applying VectorSizeHint for column '{col_name}' on full dataset...")
            return model_size_hint.transform(data)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")

    return data

# Applying the function to each vector column
for col_name in vector_cols:
    print(f"Applying VectorSizeHint to vector column: '{col_name}'")
    train_data = apply_vector_size_hint(train_data, col_name)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")
pipeline_final_assembler = Pipeline(stages=[final_assembler])
model_final_assembler = pipeline_final_assembler.fit(train_data)
print("Pipeline Processing Done")

# Step 6: Transform train, valid, and test datasets to include the final features
train_data = model_final_assembler.transform(train_data)
valid_data = model_final_assembler.transform(valid_data)
test_data = model_final_assembler.transform(test_data)
print("Pipeline Transformation Done")

▸,:,


Applying VectorSizeHint to vector column: 'race_vector'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Column 'gender_vector' vector size: 1
Fitting VectorSizeHint for column 'gender_vector' on sample of 0.01% of data...
Applying VectorSizeHint for column 'gender_vector' on full dataset...
Final input columns for feature assembly: ['race_vector', 'gender_vector', 'age_of_TBI_diagnosis', 'MedicalHistory', 'E72', 'Q02', 'S37', 'R52', 'L74', 'T50', 'S87', 'S41', 'C25', 'V58', 'B37', 'T58', 'V41', 'X34', 'C86', 'I01', 'Q11', 'M75', 'B70', 'Q52', 'W27', 'R36', 'V65', 'V28', 'I47', 'K87', 'I52', 'E86', 'Z33', 'Q23', 'Q86', 'N46', 'L62', 'V97', 'T30', 'L50', 'S56', 'H62', 'C91', 'G70', 'Q45', 'T27', 'Z77', 'S51', 'P05', 'F29', 'A83', 'F20', 'W89', 'B68', 'O72', 'X77', 'L87', 'X39', 'B42', 'R59', 'V38', 'X04', 'J35', 'V94', 'W99', 'C54', 'N88', 'F02', 'G63', 'K06', 'J22', 'D78', 'K60', 'W18', 'D21', 'H16', 'L54', 'F11', 'D04', 'H30', 'Z20', 'V73', 'G06', 'K80', 'M19', 'T07', 'W52', 'M80', 'X82', 'H65', 'Z86', 'Y75', 'E55', 'G81', 'V63', 'E87', 'D36', 'O43', 'Z51', 'L91', 'V48', 'K22', 'E73', 'D

<IPython.core.display.Javascript object>

Pipeline Transformation Done


In [35]:
# # Step 4: Select only the 'features' column (you can also include target/label column if needed)
# train_data_final = train_data.select("features")
# valid_data_final = valid_data.select("features")
# test_data_final = test_data.select("features")
# train_data.printSchema()
# valid_data.printSchema()
# test_data.printSchema()

▸,:,


In [27]:
# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions = train_data.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    train_data = train_data.repartition(num_partitions)
    print(f"DataFrame repartitioned into {num_partitions} partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")

# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions1 = valid_data.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    valid_data = valid_data.repartition(num_partitions1)
    print(f"DataFrame repartitioned into {num_partitions1} partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")
    
# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions2 = test_data.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    test_data = test_data.repartition(num_partitions2)
    print(f"DataFrame repartitioned into {num_partitions2} partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")

▸,:,


DataFrame repartitioned into 14 partitions successfully.
DataFrame repartitioned into 14 partitions successfully.
DataFrame repartitioned into 14 partitions successfully.


In [ ]:
# from pyspark.ml.feature import VectorSizeHint, StandardScaler
# from pyspark.ml import Pipeline
# from pyspark.sql.functions import col
# # Define the column name and sample fraction for determining vector size
# col_name = "features"  # Replace with the appropriate column name
# sample_fraction = 0.00001  # Further reduce the sample fraction
# sample_seed = 42  # Set a seed for reproducibility

# # Step 1: Sample a small portion of the data to determine vector size
# try:    
#     # Sample a small fraction of the data with a seed
#     sampled_df = train_data_final.select(col(col_name)).sample(False, sample_fraction, seed=sample_seed)

#     # Step 2: Get the vector size from the first row (assumes all vectors have the same size)
#     first_vector_size = sampled_df.first()[0].size
#     print(f"Column '{col_name}' vector size: {first_vector_size}")

# except Exception as e:
#     print(f"Error while sampling the DataFrame: {e}")
#     first_vector_size = None

# if first_vector_size is not None and first_vector_size > 0:
#     try:
#         # Step 3: Define VectorSizeHint and StandardScaler
#         size_hint = VectorSizeHint(inputCol=col_name, size=first_vector_size)
#         scaler = StandardScaler(inputCol=col_name, outputCol="features_scaled")

#         # Step 4: Create the pipeline with both VectorSizeHint and StandardScaler
#         pipeline_scaler = Pipeline(stages=[size_hint, scaler])

#         # Step 5: Fit the pipeline on the full training data
#         print("Fitting VectorSizeHint and StandardScaler on the data...")
#         model_scaler = pipeline_scaler.fit(train_data_final)

#         # Step 6: Transform the train, valid, and test data
#         print("Transforming train, valid, and test data with VectorSizeHint and StandardScaler...")
#         train_df = model_scaler.transform(train_data_final)
#         valid_df = model_scaler.transform(valid_data_final)
#         test_df = model_scaler.transform(test_data_final)

#         print("VectorSizeHint and StandardScaler successfully applied.")
#     except Exception as e:
#         print(f"Error during pipeline fitting and transformation: {e}")
# else:
#     print(f"Unable to determine vector size for column '{col_name}' or vector size is 0.")


▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
train_data = train_data.withColumn('label',train_data.label.cast('double'))
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

▸,:,


In [29]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Display the counts
print(f"Number of 0's in the label column: {count_zeros}")
print(f"Number of 1's in the label column: {count_ones}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of 0's in the label column: 705565
Number of 1's in the label column: 106624


<IPython.core.display.Javascript object>

In [30]:
from pyspark.sql.functions import col

# Number of 0's and 1's in the label column
count_zeros = 705565
count_ones = 106624

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Upsampling
# Calculate the number of times we need to duplicate the minority class to match the desired count
upsample_ratio = int((count_zeros - count_ones) / count_ones)
remaining_minority_samples = (count_zeros - count_ones) % count_ones
# Duplicate the minority class DataFrame
upsampled_minority_class_df = minority_class_df
for i in range(upsample_ratio):
    upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)
# Add remaining samples to reach the exact count
upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df.sample(withReplacement=True, fraction=(remaining_minority_samples / count_ones)))

# Downsampling
# Calculate the fraction for downsampling the majority class
downsample_fraction = count_ones / count_zeros
# Sample the majority class to match the number of minority class samples
downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)

# Combine the upsampled minority class with the downsampled majority class
train_data_balanced= upsampled_minority_class_df.union(downsampled_majority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 1).count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of 1's in the balanced DataFrame:  705693


<IPython.core.display.Javascript object>

In [36]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier

# Step 6: Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Step 7: Function to perform cross-validation using train and validation sets
def manual_cross_validation(train_df: DataFrame, val_df: DataFrame, param_grid: dict, label_col: str):
    results = []
    best_model = None
    best_auc = 0.0  # Track the best AUC

    for max_depth in param_grid['maxDepth']:
        for max_iter in param_grid['maxIter']:
            # Train GBT model on the train data
            gbt = GBTClassifier(featuresCol="features", labelCol=label_col,
                                maxDepth=max_depth, maxIter=max_iter)

            model = gbt.fit(train_data_balanced)

            # Validate on validation set
            val_data_pred = model.transform(valid_data)

            # Calculate AUC using the manual function
            val_auc = calculate_manual_auc(val_data_pred, label_col, "probability")

            # Calculate accuracy for logging
            val_accuracy = calculate_accuracy(val_data_pred, label_col, "prediction")

            # Store results
            results.append((max_depth, max_iter, val_accuracy, val_auc))

            # Keep track of the best model based on validation AUC
            if val_auc > best_auc:
                best_auc = val_auc
                best_model = (gbt, model, val_accuracy, val_auc)

    return results, best_model[1]  # Return results and best model

def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    # Cast label and prediction to ensure they are the same type
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))

    # Compute correct predictions and total predictions
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)  # Round accuracy to 4 decimal places

# Function to calculate AUC manually with improved precision
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns (label and the probability for the positive class)
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Assuming the second column is the positive class probability

    # Sort predictions by probability, descending
    preds = preds.sortBy(lambda x: -x[1]).collect()

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    # Initialize variables for AUC calculation
    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Add precision control by avoiding division by zero
    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative

        # Trapezoidal area for AUC calculation
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)  # Round AUC to 4 decimal places

# Step 9: Run manual cross-validation on the balanced train and validation data
cv_results, best_model = manual_cross_validation(train_data_balanced, valid_data, param_grid, 'label')

# Find the best hyperparameters based on validation AUC
best_params = max(cv_results, key=lambda x: x[3])
print(f"Best parameters: maxDepth={best_params[0]}, maxIter={best_params[1]}, Validation Accuracy={best_params[2]}, Validation AUC={best_params[3]}")

# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = best_model.transform(train_data_balanced)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
print(f"Train Accuracy: {train_accuracy}")

# Calculate AUC for train data
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")
print(f"Train AUC: {train_auc}")

# Step 9: Use the best model (already trained) to transform and evaluate on validation and test data
# Transform validation data
val_data_pred = best_model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
print(f"Validation Accuracy: {val_accuracy}")

# Calculate AUC for validation data
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")
print(f"Validation AUC: {val_auc}")

# Transform test data
test_data_pred = best_model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
print(f"Test Accuracy: {test_accuracy}")

# Calculate AUC for test data
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")
print(f"Test AUC: {test_auc}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Py4JJavaError: An error occurred while calling o647.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 5 in stage 164.0 failed 1 times, most recent failure: Lost task 5.0 in stage 164.0 (TID 2319, localhost, executor driver): java.lang.OutOfMemoryError: Java heap space

Driver stacktrace:
	at org.apache.spark.scheduler.DAGScheduler.org$apache$spark$scheduler$DAGScheduler$$failJobAndIndependentStages(DAGScheduler.scala:1889)
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$abortStage$1.apply(DAGScheduler.scala:1877)
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$abortStage$1.apply(DAGScheduler.scala:1876)
	at scala.collection.mutable.ResizableArray$class.foreach(ResizableArray.scala:59)
	at scala.collection.mutable.ArrayBuffer.foreach(ArrayBuffer.scala:48)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:1876)
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$handleTaskSetFailed$1.apply(DAGScheduler.scala:926)
	at org.apa

<IPython.core.display.Javascript object>

In [17]:
# # Function to calculate and print confusion matrix metrics
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def calculate_confusion_matrix_metrics(df: DataFrame, label_col: str, prediction_col: str):
    # Calculate confusion matrix counts
    confusion_counts = df.groupBy(label_col, prediction_col).agg(F.count("*").alias("count"))

    # Initialize metrics as DataFrames
    true_positives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_positives"))
    true_negatives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_negatives"))
    false_positives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_positives"))
    false_negatives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_negatives"))

    # Combine all metrics into one DataFrame
    metrics = true_positives.crossJoin(true_negatives).crossJoin(false_positives).crossJoin(false_negatives)

    # Calculate overall counts
    total_count = df.count()
    accuracy = (metrics.select("true_positives").first()[0] + metrics.select("true_negatives").first()[0]) / total_count if total_count > 0 else 0.0

    # Extract metric values from the DataFrame
    tp = metrics.select("true_positives").first()[0]
    tn = metrics.select("true_negatives").first()[0]
    fp = metrics.select("false_positives").first()[0]
    fn = metrics.select("false_negatives").first()[0]

    # Calculate precision, recall, F1 score for class 1
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0.0

    # Calculate precision, recall, F1 score for class 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Return all metrics in a structured format
    return {
        "metrics": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        },
        "accuracy": accuracy,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_score_1": f1_score_1,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_score_0": f1_score_0,
        "specificity": specificity
    }

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        print("Confusion Matrix Metrics:")
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

Confusion Matrix Metrics:
True Positives: 1417
True Negatives: 4194
False Positives: 6106
False Negatives: 74
Accuracy: 0.4759
Precision 1: 0.1884
Recall 1: 0.9504
F1 Score 1: 0.3144
Precision 0: 0.9827
Recall 0: 0.4072
F1 Score 0: 0.5758
Specificity: 0.4072


In [31]:
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
# Initialize the GBTClassifier
gbt = GBTClassifier(labelCol="label", featuresCol="scaledFeatures")
# Set up the parameter grid for hyperparameter tuning
paramGrid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_accuracy = 0.0
best_params = {}

# Loop through all combinations of hyperparameters
for max_depth in paramGrid["maxDepth"]:
    for max_iter in paramGrid["maxIter"]:
        # Set hyperparameters
        gbt.setMaxDepth(max_depth)
        gbt.setMaxIter(max_iter)

        # Fit the model on the training data
        model = gbt.fit(train_data)
        # *** Train data evaluation ***
        train_predictions = model.transform(train_data)

        # Calculate train AUC
        train_auc = evaluator.evaluate(train_predictions)

        # Calculate train accuracy
        predictionAndTarget_train = train_predictions.select("label", "prediction")
        predictionAndTarget_train_rdd = predictionAndTarget_train.rdd.map(tuple)
        metrics_multi_train = MulticlassMetrics(predictionAndTarget_train_rdd)
        train_accuracy = metrics_multi_train.accuracy        

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)

        # Calculate validation AUC
        validation_auc = evaluator.evaluate(valid_predictions)

        # Calculate validation accuracy
        predictionAndTarget_valid = valid_predictions.select("label", "prediction")
        predictionAndTarget_valid_rdd = predictionAndTarget_valid.rdd.map(tuple)
        metrics_multi = MulticlassMetrics(predictionAndTarget_valid_rdd)
        validation_accuracy = metrics_multi.accuracy

        # Print the current parameters, validation AUC, and validation accuracy
        print(f"MaxDepth: {max_depth}, MaxIter: {max_iter}")
        print(f"Train AUC: {train_auc}, Train Accuracy: {train_accuracy}")
        print(f"Validation AUC: {validation_auc}, Validation Accuracy: {validation_accuracy}")
        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_accuracy = validation_accuracy
            best_model = model
            best_params = {"maxDepth": max_depth, "maxIter": max_iter}

# Print the best parameters and validation AUC and Accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")
print(f"Best Validation Accuracy: {best_accuracy}")
# # Cast the label column in the test data to double (if necessary)
# test_data = test_data.withColumn('label', test_data.label.cast('double'))
# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, MaxIter: 10
Train AUC: 0.8651749697615753, Train Accuracy: 0.870610414253678
Validation AUC: 0.8569385115462247, Validation Accuracy: 0.8684774226715791
MaxDepth: 5, MaxIter: 20
Train AUC: 0.8808223765354322, Train Accuracy: 0.8707829172724808
Validation AUC: 0.8707312212088123, Validation Accuracy: 0.8685631051323794
MaxDepth: 10, MaxIter: 10
Train AUC: 0.9138084859089193, Train Accuracy: 0.9025234727322013
Validation AUC: 0.8925182298732872, Validation Accuracy: 0.897780824265273
MaxDepth: 10, MaxIter: 20
Train AUC: 0.9309862486506668, Train Accuracy: 0.9105202198181326
Validation AUC: 0.9054685437584309, Validation Accuracy: 0.9042070088252935
Best Parameters: {'maxDepth': 10, 'maxIter': 20}
Best Validation AUC: 0.9054685437584309
Best Validation Accuracy: 0.9042070088252935
Test AUC: 0.8299305587281283
Test Accuracy: 0.9045348131448886


In [20]:
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
# Initialize the GBTClassifier
gbt = GBTClassifier(labelCol="label", featuresCol="features_scaled")
# Set up the parameter grid for hyperparameter tuning
paramGrid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_accuracy = 0.0
best_params = {}

# Loop through all combinations of hyperparameters
for max_depth in paramGrid["maxDepth"]:
    for max_iter in paramGrid["maxIter"]:
        # Set hyperparameters
        gbt.setMaxDepth(max_depth)
        gbt.setMaxIter(max_iter)

        # Fit the model on the training data
        model = gbt.fit(train_data)
        # *** Train data evaluation ***
        train_predictions = model.transform(train_data)

        # Calculate train AUC
        train_auc = evaluator.evaluate(train_predictions)

        # Calculate train accuracy
        predictionAndTarget_train = train_predictions.select("label", "prediction")
        predictionAndTarget_train_rdd = predictionAndTarget_train.rdd.map(tuple)
        metrics_multi_train = MulticlassMetrics(predictionAndTarget_train_rdd)
        train_accuracy = metrics_multi_train.accuracy        

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)

        # Calculate validation AUC
        validation_auc = evaluator.evaluate(valid_predictions)

        # Calculate validation accuracy
        predictionAndTarget_valid = valid_predictions.select("label", "prediction")
        predictionAndTarget_valid_rdd = predictionAndTarget_valid.rdd.map(tuple)
        metrics_multi = MulticlassMetrics(predictionAndTarget_valid_rdd)
        validation_accuracy = metrics_multi.accuracy

        # Print the current parameters, validation AUC, and validation accuracy
        print(f"MaxDepth: {max_depth}, MaxIter: {max_iter}")
        print(f"Train AUC: {train_auc}, Train Accuracy: {train_accuracy}")
        print(f"Validation AUC: {validation_auc}, Validation Accuracy: {validation_accuracy}")
        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_accuracy = validation_accuracy
            best_model = model
            best_params = {"maxDepth": max_depth, "maxIter": max_iter}

# Print the best parameters and validation AUC and Accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")
print(f"Best Validation Accuracy: {best_accuracy}")
# # Cast the label column in the test data to double (if necessary)
# test_data = test_data.withColumn('label', test_data.label.cast('double'))
# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, MaxIter: 10
Train AUC: 0.8627592146420469, Train Accuracy: 0.8686677094751881
Validation AUC: 0.8516838969497653, Validation Accuracy: 0.8665040232836843
MaxDepth: 5, MaxIter: 20
Train AUC: 0.8817763303501394, Train Accuracy: 0.8689263588328756
Validation AUC: 0.8685932530097205, Validation Accuracy: 0.8665896250642013
MaxDepth: 10, MaxIter: 10
Train AUC: 0.9163444500049789, Train Accuracy: 0.9045091204690175
Validation AUC: 0.8876632841155289, Validation Accuracy: 0.8986474918678309
MaxDepth: 10, MaxIter: 20
Train AUC: 0.9326584681523942, Train Accuracy: 0.9108152381421586
Validation AUC: 0.9028453940036097, Validation Accuracy: 0.9026707755521315
Best Parameters: {'maxDepth': 10, 'maxIter': 20}
Best Validation AUC: 0.9028453940036097
Best Validation Accuracy: 0.9026707755521315
Test AUC: 0.8134272163197057
Test Accuracy: 0.9024960998439937


In [18]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier
from functools import reduce

# Step 6: Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Step 7: Function to perform cross-validation using train and validation sets
def manual_cross_validation(train_df: DataFrame, val_df: DataFrame, param_grid: dict, label_col: str):
    results = []
    best_model = None
    best_auc = 0.0  # Track the best AUC

    for max_depth in param_grid['maxDepth']:
        for max_iter in param_grid['maxIter']:
            # Train GBT model on the train data
            gbt = GBTClassifier(featuresCol="features_scaled", labelCol=label_col,
                                maxDepth=max_depth, maxIter=max_iter)

            model = gbt.fit(train_data)

            # Validate on validation set
            val_data_pred = model.transform(valid_data)

            # Calculate AUC using the manual function
            val_auc = calculate_manual_auc(val_data_pred, label_col, "probability")

            # Calculate accuracy for logging
            val_accuracy = calculate_accuracy(val_data_pred, label_col, "prediction")

            # Store results
            results.append((max_depth, max_iter, val_accuracy, val_auc))

            # Keep track of the best model based on validation AUC
            if val_auc > best_auc:
                best_auc = val_auc
                best_model = (gbt, model, val_accuracy, val_auc)

    return results, best_model[1]  # Return results and best model

# Function to calculate accuracy manually
def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return accuracy

# Function to calculate AUC manually
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Assuming the second column is the positive class probability

    # Sort predictions by probability
    preds = sorted(preds.collect(), key=lambda x: x[1], reverse=True)

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    previous_tpr = 0.0
    previous_fpr = 0.0

    for label, probability in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative
        
        # Calculate the area under the curve (AUC)
        auc += (fpr - previous_fpr) * (tpr + previous_tpr) / 2
        previous_tpr = tpr
        previous_fpr = fpr

    return auc

# Step 9: Run manual cross-validation on the balanced train and validation data
cv_results, best_model = manual_cross_validation(train_data, valid_data, param_grid, 'label')

# Find the best hyperparameters based on validation AUC
best_params = max(cv_results, key=lambda x: x[3])
print(f"Best parameters: maxDepth={best_params[0]}, maxIter={best_params[1]}, Validation Accuracy={best_params[2]}, Validation AUC={best_params[3]}")

# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = best_model.transform(train_data)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
print(f"Train Accuracy: {train_accuracy}")

# Calculate AUC for train data
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")
print(f"Train AUC: {train_auc}")

# Step 9: Use the best model (already trained) to transform and evaluate on validation and test data
# Transform validation data
val_data_pred = best_model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
print(f"Validation Accuracy: {val_accuracy}")

# Calculate AUC for validation data
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")
print(f"Validation AUC: {val_auc}")

# Transform test data
test_data_pred = best_model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
print(f"Test Accuracy: {test_accuracy}")

# Calculate AUC for test data
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")
print(f"Test AUC: {test_auc}")

Best parameters: maxDepth=10, maxIter=20, Validation Accuracy=0.9026707755521315, Validation AUC=0.9029568928430726
Train Accuracy: 0.9108152381421586
Train AUC: 0.9327731412290579
Validation Accuracy: 0.9026707755521315
Validation AUC: 0.9029568928430726
Test Accuracy: 0.9024960998439937
Test AUC: 0.9082155148346786


In [33]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier

# Step 6: Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Step 7: Function to perform cross-validation using train and validation sets
def manual_cross_validation(train_df: DataFrame, val_df: DataFrame, param_grid: dict, label_col: str):
    results = []
    best_model = None
    best_auc = 0.0  # Track the best AUC

    for max_depth in param_grid['maxDepth']:
        for max_iter in param_grid['maxIter']:
            # Train GBT model on the train data
            gbt = GBTClassifier(featuresCol="scaledFeatures", labelCol=label_col,
                                maxDepth=max_depth, maxIter=max_iter)

            model = gbt.fit(train_df)

            # Validate on validation set
            val_data_pred = model.transform(val_df)

            # Calculate AUC using the manual function
            val_auc = calculate_manual_auc(val_data_pred, label_col, "probability")

            # Calculate accuracy for logging
            val_accuracy = calculate_accuracy(val_data_pred, label_col, "prediction")

            # Store results
            results.append((max_depth, max_iter, val_accuracy, val_auc))

            # Keep track of the best model based on validation AUC
            if val_auc > best_auc:
                best_auc = val_auc
                best_model = (gbt, model, val_accuracy, val_auc)

    return results, best_model[1]  # Return results and best model

def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    # Cast label and prediction to ensure they are the same type
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))

    # Compute correct predictions and total predictions
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)  # Round accuracy to 4 decimal places

# Function to calculate AUC manually with improved precision
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns (label and the probability for the positive class)
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Assuming the second column is the positive class probability

    # Sort predictions by probability, descending
    preds = preds.sortBy(lambda x: -x[1]).collect()

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    # Initialize variables for AUC calculation
    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Add precision control by avoiding division by zero
    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative

        # Trapezoidal area for AUC calculation
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)  # Round AUC to 4 decimal places

# Step 9: Run manual cross-validation on the balanced train and validation data
cv_results, best_model = manual_cross_validation(train_data, valid_data, param_grid, 'label')

# Find the best hyperparameters based on validation AUC
best_params = max(cv_results, key=lambda x: x[3])
print(f"Best parameters: maxDepth={best_params[0]}, maxIter={best_params[1]}, Validation Accuracy={best_params[2]}, Validation AUC={best_params[3]}")

# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = best_model.transform(train_data)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
print(f"Train Accuracy: {train_accuracy}")

# Calculate AUC for train data
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")
print(f"Train AUC: {train_auc}")

# Step 9: Use the best model (already trained) to transform and evaluate on validation and test data
# Transform validation data
val_data_pred = best_model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
print(f"Validation Accuracy: {val_accuracy}")

# Calculate AUC for validation data
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")
print(f"Validation AUC: {val_auc}")

# Transform test data
test_data_pred = best_model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
print(f"Test Accuracy: {test_accuracy}")

# Calculate AUC for test data
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")
print(f"Test AUC: {test_auc}")

Best parameters: maxDepth=10, maxIter=20, Validation Accuracy=0.9038, Validation AUC=0.9057
Train Accuracy: 0.9105
Train AUC: 0.9312
Validation Accuracy: 0.9038
Validation AUC: 0.9057
Test Accuracy: 0.9055
Test AUC: 0.9024
